# Notebook 06 - Comparacion de Estrategias

## Objetivos
- Comparar zero-shot, few-shot y CoT en la misma tarea.
- Evaluar calidad con criterios simples.
- Visualizar resultados en tablas y graficos.

## Introduccion
La mejor forma de aprender prompt engineering es comparar. Este notebook ejecuta la misma tarea con 4 estrategias y evalua los resultados.

In [ ]:
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('..')
DATASETS = BASE / 'datasets'
print('Entorno listo. Datasets:', list(DATASETS.glob('*.csv')))

In [ ]:
from transformers import pipeline, set_seed

set_seed(42)
generator = pipeline('text-generation', model='datificate/gpt2-small-spanish')
print('GPT-2 en español listo para experimentos de prompting')

In [ ]:
def generar(prompt, max_new_tokens=40, temperature=0.7):
    out = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        pad_token_id=generator.tokenizer.eos_token_id,
    )
    return out[0]['generated_text']

print('Funcion generar() lista')

## 1) Tareas del curso

In [ ]:
df_tareas = pd.read_csv(DATASETS / 'tareas_comparacion.csv')
display(df_tareas)

## 2) Funcion de evaluacion simple

In [ ]:
def evaluar_respuesta(respuesta, criterios):
    scores = {}
    resp_lower = respuesta.lower()
    scores['tiene_contenido'] = 1 if len(respuesta.strip()) > 10 else 0
    scores['no_muy_largo'] = 1 if len(respuesta) < 500 else 0
    if 'keyword' in criterios:
        scores['keyword_presente'] = 1 if criterios['keyword'] in resp_lower else 0
    return scores

print('Funcion evaluar_respuesta() lista')

## 3) Experimento: clasificar urgencia con 4 estrategias

In [ ]:
df_tickets = pd.read_csv(DATASETS / 'tickets_soporte.csv')
ticket_test = df_tickets.iloc[5]['ticket']
urgencia_real = df_tickets.iloc[5]['urgencia']

estrategias = {
    'zero-shot': f'Clasifica urgencia alta/media/baja. Solo etiqueta:\nTicket: {ticket_test}\nUrgencia:',
    'one-shot': f'Clasifica urgencia.\nTicket: Sistema caido\nUrgencia: alta\nTicket: {ticket_test}\nUrgencia:',
    'few-shot': '\n'.join([
        'Clasifica urgencia alta/media/baja. Solo etiqueta:',
        *[f'Ticket: {r["ticket"]}\nUrgencia: {r["urgencia"]}' for _, r in df_tickets.head(3).iterrows()],
        f'Ticket: {ticket_test}\nUrgencia:',
    ]),
    'chain-of-thought': f'Ticket: {ticket_test}\nPiensa paso a paso: 1) ¿Cuál es el problema? 2) ¿Cuántos usuarios afectados? 3) ¿Nivel de urgencia?\nAnalisis:',
}

resultados = []
for nombre, prompt in estrategias.items():
    resp = generar(prompt, max_new_tokens=30 if nombre == 'chain-of-thought' else 5, temperature=0.3)
    resultados.append({
        'estrategia': nombre,
        'ticket': ticket_test[:40],
        'urgencia_real': urgencia_real,
        'respuesta': resp[-40:],
    })

df_resultados = pd.DataFrame(resultados)
display(df_resultados)

## 4) Experimento: misma tarea con distintas temperaturas

In [ ]:
prompt_base = 'Resume en una oracion: Las ventas del Q3 alcanzaron 2.4M USD con 18% de crecimiento.\nResumen:'
temp_results = []
for temp in [0.2, 0.5, 0.8, 1.2]:
    r = generar(prompt_base, max_new_tokens=30, temperature=temp)
    temp_results.append({'temperatura': temp, 'resumen': r[-60:]})
display(pd.DataFrame(temp_results))

## 5) Visualizacion comparativa

In [ ]:
estrategias_count = df_resultados['estrategia'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df_resultados['estrategia'].plot(kind='barh', ax=axes[0], color='coral')
axes[0].set_title('Estrategias evaluadas')
axes[0].set_xlabel('Cantidad')

temps = [r['temperatura'] for r in temp_results]
lens = [len(r['resumen']) for r in temp_results]
axes[1].bar(range(len(temps)), lens, color='steelblue')
axes[1].set_xticks(range(len(temps)))
axes[1].set_xticklabels([str(t) for t in temps])
axes[1].set_title('Longitud de respuesta por temperatura')
axes[1].set_xlabel('Temperatura')
axes[1].set_ylabel('Caracteres')

plt.tight_layout()
plt.show()

## 6) Matriz estrategia vs tipo de tarea

In [ ]:
matriz = pd.DataFrame({
    'zero-shot': ['Alta', 'Media', 'Alta', 'Alta', 'Media'],
    'one-shot': ['Media', 'Alta', 'Alta', 'Media', 'Alta'],
    'few-shot': ['Baja', 'Alta', 'Alta', 'Media', 'Alta'],
    'chain-of-thought': ['Baja', 'Media', 'Baja', 'Baja', 'Alta'],
    'role': ['Media', 'Media', 'Media', 'Alta', 'Alta'],
    'context': ['Baja', 'Baja', 'Alta', 'Baja', 'Alta'],
}, index=['Clasificacion simple', 'Categorias propias', 'Resumen', 'Traduccion', 'Razonamiento'])
display(matriz)

## Resultados
Comparamos 4 estrategias de prompting en la misma tarea y evaluamos el impacto de la temperatura en la longitud de respuesta.

## Conclusiones
No hay estrategia ganadora universal. Few-shot brilla con categorias propias; CoT con razonamiento; context con documentos. Evaluar siempre.

## Ejercicios guiados resueltos
**Ejercicio:** Ejecuta las 4 estrategias en otro ticket del CSV.

**Solucion:**

In [ ]:
otro = df_tickets.iloc[3]
for nombre, fn in [('zs', lambda t: f'Urgencia alta/media/baja: {t}\nEtiqueta:')]:
    print(generar(fn(otro['ticket']), max_new_tokens=5, temperature=0.3)[-20:])

## Ejercicios propuestos
1. Evalua 10 tickets con few-shot y calcula accuracy.
2. Compara role prompting con 3 roles en la misma tarea.
3. Crea tu propia matriz estrategia vs tarea.

## Preguntas de reflexion
1. Como definirias 'calidad' de un prompt objetivamente?
2. Cuando la comparacion side-by-side es engañosa?
3. Que metricas usarias en produccion?